# 03 · Panel c: PICALM exon 13 and the 24-nt exon, sashimi plots and PSI

Fourth of six notebooks (00 screen and locus → 01 download → 02 rMATS → **03 panel c** → 04 eCLIP and rbp-maps → 05 optional window maps). Inputs: the BAMs from notebook 01 and the rMATS skipped-exon tables from notebook 02.

Panel c shows how QKI knockdown changes the inclusion of two PICALM exons in K562 and HepG2: exon 13 (150 nt, in the MANE transcript NM_007166) and a 24-nt exon (in NM_001008660). This notebook rebuilds it from the rMATS output in three steps:

1. Pull each event's row out of the rMATS table into a one-event file.
2. Draw sashimi plots with [rmats2sashimiplot](https://github.com/Xinglab/rmats2sashimiplot): RNA-seq coverage plus junction arcs labelled with read counts, control vs knockdown (replicates averaged), and one track per replicate.
3. Plot PSI per replicate with ΔPSI and FDR from rMATS, and the knockdown level from ENCODE's DESeq2 tables.

Expected (ENCODE release), exon 13: QKI knockdown 26% → 48% in K562 (ΔPSI +0.22) and 20% → 68% in HepG2 (+0.48). 24-nt exon: QKI knockdown 15% → 3% in K562 (−0.12) and 74% → 21% in HepG2 (−0.53).

**Two things to know when reading the sashimi plot.**

- PICALM is on the minus strand. rmats2sashimiplot draws minus-strand events 5′ → 3′ (genomic coordinates decrease left to right), so the exon upstream in the transcript is on the left.
- rmats2sashimiplot counts junction reads its own way, so its arc labels can differ slightly from rMATS's IJC/SJC columns ([issue 33](https://github.com/Xinglab/rmats2sashimiplot/issues/33)). The PSI and FDR values in step 3 come from rMATS.

## Accessing RBP-ARK data through Globus

RBP-ARK keeps every Skipper output on one Globus collection. Public files can be downloaded straight over HTTPS with no login; bulk copies go through Globus.

| | |
| --- | --- |
| Globus collection | **RBP-ARK**, `9cd9be17-cbed-4b34-ab76-704b2fd6ac7b` ([open in the Globus web app](https://app.globus.org/file-manager?origin_id=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b&origin_path=%2F)) |
| HTTPS address of the same files | `https://g-f8db23.0ed28.75bc.data.globus.org/<path>` |

**Where things are.** `<version>/<run>` is the Skipper version and run folder. Since November 2026 every ENCODE3, ENCORE and ENCORE2 dataset on rbp-ark.com (gene search, dataset pages, BDBags) is Skipper `2.0.0/v2`, and so are these notebooks. Earlier runs (`1.99.0/v1`, `1.100.0/v1`) remain on the collection.

| Path on the collection | Contents |
| --- | --- |
| `/encode3/eclip/Skipper/<version>/<run>/output/` | ENCODE3 eCLIP datasets, named `<RBP>_<cell>_ENCSR…` |
| `/encore2/eclip/Skipper/<version>/<run>/output/` | ENCORE2 eCLIP datasets with 5000-series IDs (e.g. `QKI_K562_5007`) |
| `/encore/eclip/Skipper/<version>/<run>/output/` | ENCORE eCLIP datasets with 4000-series IDs |
| `/summary_data/` | summary tables (QC, window counts) |

Inside each `output/` folder, files are grouped by type, then named by dataset:

| Subfolder | Files |
| --- | --- |
| `reproducible_enriched_windows/` | `<dataset>.reproducible_enriched_windows.tsv.gz`, the enriched windows |
| `reproducible_enriched_re/` | enriched repetitive elements |
| `secondary_results/bams/dedup/genome_R2/` | deduplicated, read-2-only genome BAMs (ENCODE3) |
| `secondary_results/bams/dedup/genome/` | deduplicated genome BAMs (ENCORE) |
| `secondary_results/bams/raw/genome/` | alignments before deduplication |
| `secondary_results/bigwigs/scaled/plus/`, `.../minus/` | strand-specific signal tracks |
| `QC/`, `gene_sets/`, `homer/` | QC reports, gene set enrichment, motifs |

**Three ways to get files.**

1. **One dataset: BDBag.** Each dataset page on rbp-ark.com has a **Download BDBag** link. The bag's `fetch.txt` lists every file of that dataset with its HTTPS URL, size and path, and `manifest-sha256.txt` its checksum. Notebook 04 reads the same file list from the RBP-ARK API and downloads only the BAMs it needs.
2. **Many datasets: Globus web app.** Sign in at [app.globus.org](https://app.globus.org) with your institution or a free Globus ID, open the RBP-ARK collection (link above), select folders or files, and transfer them to a collection you can write to: your laptop via [Globus Connect Personal](https://www.globus.org/globus-connect-personal), or your institution's cluster or storage collection.
3. **Scripted: Globus CLI.**

```bash
pip install globus-cli
globus login                                   # once; opens a browser
RBPARK=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b
OUT=/encode3/eclip/Skipper/2.0.0/v2/output

# list what is there
globus ls "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/" | grep QKI

# copy one file to this computer (Globus Connect Personal must be running)
DEST=$(globus endpoint local-id)
globus transfer "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" \
    "$DEST:$HOME/rbpark/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" --label "QKI eCLIP"

# or copy a whole folder
globus transfer --recursive "$RBPARK:$OUT/reproducible_enriched_windows/" "$DEST:$HOME/rbpark/windows/"
globus task list                               # follow progress
```

If the CLI asks for extra consent the first time it touches the collection, it prints the exact `globus session consent ...` command to run. The Globus CLI commands are shown for reference and were not run in testing; the BDBag and HTTPS routes were. Please copy only what you need: the public HTTPS endpoint is shared, and bulk sweeps have taken it offline.

**In this notebook:** nothing comes from RBP-ARK; panel c uses ENCODE knockdown data. RBP-ARK data is used in notebooks 00, 04 and 05.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymupdf
from IPython.display import Image, display

sys.path.insert(0, ".")
import cs3_utils as U

ROOT, DATA = U.folders()
BAM_DIR = DATA / "downloads" / "shrna_bams_hg19"
REF_DIR = DATA / "downloads" / "encode_reference"
RMATS_DIR = DATA / "rmats"
OUT = ROOT / "figures" / "panel_c"
OUT.mkdir(parents=True, exist_ok=True)

# "rerun" = notebook 02's rMATS-turbo output; "encode" = ENCODE's released rMATS 3.2.1 output (used in the draft figure)
SOURCE = os.environ.get("SE_SOURCE", "rerun")
PLOTS = [("QKI_K562", "exon13"), ("QKI_HepG2", "exon13"), ("QKI_K562", "exon24nt"), ("QKI_HepG2", "exon24nt")]
COLORS = {"control": "#b4b3ad", "QKI": "#7a3db8"}

assert shutil.which("rmats2sashimiplot"), "rmats2sashimiplot is not on PATH"
manifest = pd.read_csv(BAM_DIR / "manifest.tsv", sep="\t")


def se_path(key):
    """Returns the rMATS skipped-exon table for knockdown `key`: notebook 02's rerun, or ENCODE's release when
    SE_SOURCE=encode.
    """
    if SOURCE == "rerun":
        return RMATS_DIR / "rerun" / key / "SE.MATS.JC.txt"
    return RMATS_DIR / "encode_release" / key / "SE.MATS.JunctionCountOnly.txt"


for key in U.KNOCKDOWNS:
    print(key, se_path(key), "exists" if se_path(key).exists() else "MISSING (run notebook 02)")

## 1. One-event rMATS files

hg19 coordinates (0-based start, as rMATS writes them; table in notebook 02). rmats2sashimiplot plots every row of the file it is given, so the row is copied, unchanged, into a file of its own. If an exon appears with other flanking exons instead, the row with the most junction reads is used and the cell says so.

In [ ]:
def write_event_file(key, exon):
    """Writes the rMATS row of PICALM `exon` in knockdown `key` to a one-event file for rmats2sashimiplot; returns the
    file and the row.
    """
    lines = se_path(key).read_text().splitlines()
    header = lines[0].split("\t")
    want = U.PICALM_HG19[exon]
    cols = {k: header.index(k) for k in want}          # first match, so the leading ID column is skipped
    rows = [line for line in lines[1:] if all(line.split("\t")[cols[k]] == str(v) for k, v in want.items())]
    if len(rows) != 1:
        same_exon = [line for line in lines[1:] if all(line.split("\t")[cols[k]] == str(want[k])
                                                       for k in ("chr", "strand", "exonStart_0base", "exonEnd"))]
        assert same_exon, "{} {}: exon not in the rMATS table".format(key, exon)
        reads = lambda line: sum(float(x) for c in ("IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2")
                                 for x in line.split("\t")[header.index(c)].split(",") if x not in ("NA", ""))
        rows = [max(same_exon, key=reads)]
        print("{} {}: flanks differ from the table above; using the row with the most junction reads ({} candidates)"
              .format(key, exon, len(same_exon)))
    out = OUT / key / "PICALM_{}.SE.txt".format(exon)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(lines[0] + "\n" + rows[0] + "\n")
    return out, dict(zip(header, rows[0].split("\t")))


events = {(key, exon): write_event_file(key, exon) for key, exon in PLOTS}
pd.DataFrame({"{} {}".format(*k): row for k, (_, row) in events.items()}).loc[
    ["geneSymbol", "upstreamES", "downstreamEE", "IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2",
     "IncLevel1", "IncLevel2", "IncLevelDifference", "PValue", "FDR"]]

## 2. Sashimi plots

Sample 1 is the knockdown and sample 2 the control, matching rMATS. A group file averages the two replicates of each condition into one track (control first); a second run draws all four replicates.

| Option | Value | Meaning |
| --- | --- | --- |
| `--event-type SE -e` | the one-event file | plot this rMATS skipped-exon event |
| `--exon_s 1 --intron_s 5` | | introns drawn at 1/5 scale so short exons are visible |
| `--group-info` | `Control: 3-4`, `<RBP> knockdown: 1-2` | indices into `--b1` then `--b2` |
| `--min-counts 5` | | hides junctions with fewer than 5 reads |
| `--keep-event-chr-prefix` | | ENCODE's BAMs list ERCC spike-in contigs first, which makes rmats2sashimiplot think chromosomes lack the `chr` prefix |

In [ ]:
def run_sashimi(key, exon, grouped=True):
    """Runs rmats2sashimiplot for PICALM `exon` in knockdown `key`, replicates averaged (`grouped`) or one track each,
    and returns the plot's PDF.
    """
    rbp = U.KNOCKDOWNS[key]["rbp"]
    sub = manifest[manifest["knockdown"] == key].sort_values("rep")
    kd = sub.loc[sub["condition"] == "knockdown", "bam"].tolist()
    ctl = sub.loc[sub["condition"] == "control", "bam"].tolist()
    event_file, _ = events[(key, exon)]
    out = OUT / key / "{}_{}".format(exon, "sashimi_grouped" if grouped else "sashimi_replicates")
    cmd = ["rmats2sashimiplot", "--b1", ",".join(kd), "--b2", ",".join(ctl),
           "--event-type", "SE", "-e", str(event_file),
           "--l1", rbp + "_KD", "--l2", "Control",
           "--exon_s", "1", "--intron_s", "5", "--min-counts", "5",
           "--font-size", "8", "--fig-width", "7", "--keep-event-chr-prefix", "-o", str(out)]
    if grouped:
        gf = OUT / key / "grouping.gf"
        gf.write_text("Control: 3-4\n{} knockdown: 1-2\n".format(rbp))
        cmd += ["--group-info", str(gf), "--color", "{},{}".format(COLORS["control"], COLORS[rbp]), "--fig-height", "3.5"]
    else:
        cmd += ["--color", ",".join([COLORS[rbp]] * 2 + [COLORS["control"]] * 2), "--fig-height", "6"]
    shutil.rmtree(out, ignore_errors=True)   # an index left by an earlier run makes rmats2sashimiplot skip re-indexing
    out.mkdir(parents=True, exist_ok=True)
    (out / "command.txt").write_text(" ".join(cmd) + "\n")
    with open(out / "rmats2sashimiplot.log", "w") as log:
        status = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT)
    assert status == 0, "rmats2sashimiplot failed; see " + str(out / "rmats2sashimiplot.log")
    pdfs = sorted((out / "Sashimi_plot").glob("*.pdf"))
    assert pdfs, "no sashimi plot written in " + str(out)
    return pdfs[0]


def pdf_to_png(pdf, dpi=200):
    """Saves the first page of `pdf` as a PNG at `dpi` and returns its path."""
    png = pdf.with_suffix(".png")
    page = pymupdf.open(pdf)[0]
    page.get_pixmap(dpi=dpi).save(png)
    return png


sashimi = {}
for key, exon in PLOTS:
    sashimi[(key, exon)] = pdf_to_png(run_sashimi(key, exon, grouped=True))
    replicates_png = pdf_to_png(run_sashimi(key, exon, grouped=False))
    print(key, exon, "- grouped:", sashimi[(key, exon)].name, "| per replicate:", replicates_png.name)
    display(Image(str(sashimi[(key, exon)]), width=700))

## 3. PSI per replicate, ΔPSI, FDR and knockdown level

ENCODE released two DESeq2 tables per knockdown. This case study uses the one without batch normalization for the knockdown level: QKI log2 fold change −0.85 (K562) and −3.36 (HepG2). The batch-normalized tables agree (−1.21, −6.06). Both are printed; the panel labels the table without batch normalization.

In [ ]:
def log2fc(acc, gene_id):
    """Returns the log2 fold change of gene `gene_id` in the ENCODE DESeq2 table with accession `acc` (NaN if the gene is
    absent).
    """
    table = pd.read_csv(REF_DIR / (acc + ".tsv"), sep="\t", index_col=0)
    hit = [i for i in table.index if str(i).startswith(gene_id + ".") or str(i) == gene_id]
    return float(table.loc[hit[0], "log2FoldChange"]) if hit else np.nan


def floats(text):
    """Returns the numbers in a comma-separated rMATS field `text`, skipping NA."""
    return [float(x) for x in str(text).split(",") if x not in ("NA", "")]


def fdr_label(fdr):
    """Returns `fdr` as a label: "FDR < 1e-10" or, for example, "FDR = 1.2e-05"."""
    return "FDR < 1e-10" if fdr < 1e-10 else "FDR = {:.1e}".format(fdr)


stats = []
for (key, exon), (_, row) in events.items():
    kd = U.KNOCKDOWNS[key]
    stats.append({"knockdown": key, "exon": exon, "PSI_control": floats(row["IncLevel2"]), "PSI_KD": floats(row["IncLevel1"]),
                  "dPSI": float(row["IncLevelDifference"]), "FDR": float(row["FDR"]),
                  "IJC_KD": row["IJC_SAMPLE_1"], "SJC_KD": row["SJC_SAMPLE_1"],
                  "IJC_control": row["IJC_SAMPLE_2"], "SJC_control": row["SJC_SAMPLE_2"],
                  "log2FC": log2fc(kd["deseq2"], kd["gene_id"]),
                  "log2FC_batch_normalized": log2fc(kd["deseq2_batch"], kd["gene_id"])})
stats = pd.DataFrame(stats)
stats.to_csv(ROOT / "results" / "panel_c_stats_{}.tsv".format(SOURCE), sep="\t", index=False)
(ROOT / "figures" / "data" / "picalm_knockdown_{}.json".format(SOURCE)).write_text(stats.to_json(orient="records", indent=1))
stats

In [ ]:
def psi_axis(ax):
    """Draws PSI per replicate (control gray, knockdown purple) for each row of `stats` on `ax`, with dPSI, FDR and the
    knockdown level.
    """
    for i, s in stats.iterrows():
        rbp = U.KNOCKDOWNS[s["knockdown"]]["rbp"]
        x0 = i * 2.6 + (0.8 if s["exon"] == "exon24nt" else 0)
        for j, (color, psi) in enumerate(((COLORS["control"], s["PSI_control"]), (COLORS[rbp], s["PSI_KD"]))):
            ax.bar(x0 + j, np.mean(psi), width=0.8, color=color)
            ax.scatter([x0 + j - 0.12, x0 + j + 0.12][:len(psi)], psi, s=14, color="#2b2b2b", zorder=3)
        ax.text(x0 + 0.5, 1.0, "{:+.2f}\n{}".format(s["dPSI"], fdr_label(s["FDR"])), ha="center", va="bottom", fontsize=7)
        ax.text(x0 + 0.5, -0.08, "{} KD\n{}\nlog2FC {:.2f}".format(rbp, s["knockdown"].split("_")[1], s["log2FC"]),
                ha="center", va="top", fontsize=7, transform=ax.get_xaxis_transform())
    ax.axvline(2 * 2.6 - 0.4, color="#e3e2de", lw=0.8)
    ax.text(0.5 * 2.6 + 0.5, 1.17, "exon 13", ha="center", fontsize=9, transform=ax.get_xaxis_transform())
    ax.text(2.5 * 2.6 + 1.3, 1.17, "24-nt exon", ha="center", fontsize=9, transform=ax.get_xaxis_transform())
    ax.set_xticks([])
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("exon inclusion (PSI)")
    ax.spines[["top", "right"]].set_visible(False)


fig = plt.figure(figsize=(14, 10))
grid = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.9], hspace=0.25, wspace=0.05, top=0.94)
for k, (key, exon) in enumerate(PLOTS):
    ax = fig.add_subplot(grid[k // 2, k % 2])
    ax.imshow(plt.imread(sashimi[(key, exon)]))
    ax.set_axis_off()
    ax.set_title("{} · {} (replicates averaged)".format(key.replace("_", " knockdown, "), exon), fontsize=9, loc="left")
psi_axis(fig.add_subplot(grid[2, :]))
fig.suptitle("Panel c · PICALM exon 13 and 24-nt exon · rMATS source: " + SOURCE + " · log2FC: DESeq2 without batch normalization",
             fontsize=10, x=0.01, y=0.985, ha="left")
for ext in ("pdf", "png"):
    fig.savefig(OUT / "panel_c_PICALM.{}".format(ext), dpi=300, bbox_inches="tight")
plt.show()
print("saved", OUT / "panel_c_PICALM.pdf")

**Check against the draft figure.** With `SE_SOURCE=encode` the PSI values, ΔPSI and junction counts match the draft figure's panel c exactly (for example QKI K562 exon 13: knockdown IJC 789,901 / SJC 422,493; control IJC 368,512 / SJC 540,719). With `SE_SOURCE=rerun` they should agree to within a few reads; notebook 02 shows how closely.